In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
print(len(documents))

72


In [4]:
import minsearch

index = minsearch.Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

results = index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [5]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def build_context(results):
    context = ""
    for r in results:
        context += f"filename: {r['filename']}\ncontent: {r['content']}\n\n"
    return context.strip()

def rag(query):
    results = index.search(query, num_results=5)
    context = build_context(results)
    prompt = f"""You are a course teaching assistant. Answer the question using the context below.

Context:
{context}

Question: {query}
""".strip()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.choices[0].message.content
    usage = response.usage
    return answer, usage

answer, usage = rag("How does the agentic loop keep calling the model until it stops?")
print("Answer:", answer[:200])
print("Input tokens:", usage.prompt_tokens)

Answer: The agentic loop keeps calling the model until it stops by employing a `while` loop that continuously checks whether the model has made a function call in its response. Here's a breakdown of how it wo
Input tokens: 7099


In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(len(chunks))

295
